# PlotPy — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loukesio/dataviz-genomicsdata/blob/Python_2026/plotpy/notebooks/00_colab_quickstart.ipynb)

PlotPy is an LLM-driven plotting agent for the **Genomics Viz with Python** course.  This notebook walks you through using it on Google Colab end-to-end: install, key, first plot, your own data.

**Run each cell in order, top to bottom.**  Where a step is non-obvious it's flagged with a ⚠️ banner.

---
## 0. Get a free Groq API key (one time, ~30 seconds)

PlotPy uses [Groq](https://console.groq.com) by default — free tier, fast, no credit card.

1. Open **[console.groq.com](https://console.groq.com)** and sign up.
2. Click **API Keys → Create API Key**.
3. Copy the `gsk_...` string somewhere safe (you'll paste it into Cell 2 below).

> *Keep the key private — don't commit it to GitHub or share it in screenshots.*

---
## 1. Install PlotPy (Cell 1)

Installs the agent + all optional plotting packages (`scikit-learn`, `pywaffle`, `squarify`, `ternary-diagram`) directly from GitHub.  Takes ~1 minute the first time.

In [ ]:
!pip install -q "plotpy[extras] @ git+https://github.com/loukesio/dataviz-genomicsdata.git@Python_2026#subdirectory=plotpy"

> **Already installed once and want to update?** Pip caches GitHub installs, so a plain re-install does nothing.  Force a fresh download with the cell below (then restart the runtime as in step 2).

In [ ]:
# Only run this cell if you've installed PlotPy before and want the latest commit.
!pip install -q --upgrade --force-reinstall --no-deps \
    "git+https://github.com/loukesio/dataviz-genomicsdata.git@Python_2026#subdirectory=plotpy"

---
## 2. ⚠️ Restart the runtime

**Runtime → Restart session** (or press `Ctrl+M` then `.`).

This is required after the first install — otherwise the old `plotpy` is already cached in memory and Python won't pick up the new module.  Skipping this step is the #1 source of `AttributeError: module 'plotpy' has no attribute 'datasets'`.

After the runtime restarts, continue from Cell 2 below — you do **not** need to re-run the install cell.

---
## 3. Import PlotPy and set your key (Cell 2)

Paste the `gsk_...` key from step 0 into the call below.

In [ ]:
import plotpy

plotpy.set_key("gsk_PASTE_YOUR_KEY_HERE")

print("plotpy version:", plotpy.__version__)
print("datasets available:", hasattr(plotpy, "datasets"))

---
## 4. Your first plot (Cell 3)

**Strict mode is the safe path** — it uses the verbatim course template, no LLM freelancing.  Run this first to confirm everything works.

In [ ]:
df = plotpy.datasets.expression()   # synthetic time-course (gene, time, tpm, sem)
df.head()

In [ ]:
plotpy.timecourse(df)               # per-plot wrapper — no LLM round-trip

Now let the agent **pick** the chart for you from a natural-language prompt:

In [ ]:
res = plotpy.ask(df, "Show how expression changes over time, with uncertainty.")
print("chosen:      ", res.chosen)
print("alternatives:", res.alternatives)
res.plot

The raw code the LLM produced is available too — useful for teaching and for debugging:

In [ ]:
print(res.code)

---
## 5. Try the rest of the catalog (Cell 4)

Each generator in `plotpy.datasets` is paired with a catalog entry.  Browse the menu:

In [ ]:
plotpy.datasets.list_datasets()

In [ ]:
# Interactive Manhattan (plotly — hover the dots)
plotpy.manhattan(plotpy.datasets.gwas(), interactive=True)

In [ ]:
# Volcano plot (DESeq2-shaped data)
plotpy.volcano(plotpy.datasets.deseq2())

In [ ]:
# Clustered heatmap — expression_matrix() returns (expr, meta); heatmap wants expr.
expr, meta = plotpy.datasets.expression_matrix()
plotpy.heatmap(expr)

---
## 6. Bring your own data

Two parts: get your file into Colab, then either let the agent explore it or force a specific chart.

**Upload your CSV — drag and drop:**

1. Click the **folder icon** in the left sidebar of Colab.
2. Drag the `.csv` from your computer into the file list — it lands at `/content/your_file.csv`.

In [ ]:
import pandas as pd

# Change the path to whatever you uploaded.  Falls back to the synthetic
# example so the cell still runs out of the box.
try:
    df = pd.read_csv("/content/your_file.csv")
except FileNotFoundError:
    df = plotpy.datasets.expression()

print("shape:  ", df.shape)
print("columns:", list(df.columns))
df.head()

**Or — file-picker widget (no sidebar fiddling):**

Uncomment to open a native file picker; choose your CSV, the next cell reads it.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(next(iter(uploaded)))
# df.head()

### Let the agent explore the data

`plotpy.ask` picks the chart for you from a natural-language prompt — useful when you don't yet know what the data wants to look like.

In [ ]:
res = plotpy.ask(df, "Explore this dataset — pick the best chart for it.")
print("chosen:      ", res.chosen)
print("alternatives:", res.alternatives)
res.plot

Re-run with different prompts to see what the agent suggests:

In [ ]:
plotpy.ask(df, "Compare distributions across groups.").plot

In [ ]:
plotpy.ask(df, "Show me how my measurement changes across conditions.").plot

### Force a specific chart — e.g. a time-course

Works directly if your columns match the course schema (`gene, time, tpm, sem`):

In [ ]:
plotpy.timecourse(df)

If your columns are named differently, rename them to match:

In [ ]:
# Adjust the LEFT-HAND side to match your column names.
df_renamed = df.rename(columns={
    "feature":    "gene",   # your category column
    "hour":       "time",   # your time column
    "expression": "tpm",    # your measurement
    "stderr":     "sem",    # your uncertainty
})
# plotpy.timecourse(df_renamed)

…or let the LLM adapt your columns on the fly with `mode="loose"`:

In [ ]:
plotpy.ask(df, "Plot a time-course line, one line per category, with error bars.", mode="loose").plot

**Tip:** `plotpy.datasets.list_datasets()` shows the schema each per-plot wrapper expects — handy for knowing what to rename to.

---
## If something breaks

| Symptom                                                                  | Fix                                                                                                                              |
| ------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------------------- |
| `AttributeError: module 'plotpy' has no attribute 'datasets'`            | Old install cached.  Run the `--force-reinstall` cell in step 1, then **Restart runtime**.                                       |
| `NameError` from generated code in `mode="loose"`                        | LLMs are stochastic — re-run, or fall back to `mode="strict"`.  Inspect what the LLM wrote with `agent.last_code` (below).        |
| `ModuleNotFoundError: pywaffle / squarify / ternary_diagram / sklearn`   | Reinstall with the `[extras]` form from step 1.                                                                                  |
| `Could not parse plot-selection JSON`                                    | Free Groq model occasionally drops the JSON envelope.  Re-run, or pass `plot_type="..."` to skip the LLM selection step.        |

### Debug what the LLM actually wrote

The `PlotAgent` is the lower-level handle — it carries `last_prompt`, `last_raw`, and `last_code` after every call.  Reach for it when `plotpy.ask()` returns something surprising.

In [ ]:
agent = plotpy.PlotAgent().inspect(plotpy.datasets.coexpression())
agent.ask("Two genes, colour by tissue.", mode="loose")

print("--- last prompt ---\n", agent.last_prompt, sep="")
print("\n--- last code ---\n", agent.last_code,   sep="")

---
## Where next

- **Catalog** — `plotpy.list_plots(day=1)` shows every chart the agent knows.
- **Strict vs loose** — see the README section *Strict vs Loose* for a side-by-side.
- **Course repo** — [github.com/loukesio/dataviz-genomicsdata](https://github.com/loukesio/dataviz-genomicsdata).
- **R sibling** — same architecture, different language: [github.com/loukesio/PlotR](https://github.com/loukesio/PlotR).